# 台股 ML 預測 — 回測

**流程**
```
載入模型 & 特徵資料
  → 全期推論（取得每個 rev_date 每支股票的預測機率）
  → 轉成 FinlabDataFrame（datetime × instrument）
  → 選前 N 支（做多）/ 後 N 支（放空）建立 position
  → backtest.sim() 回測
  → 比較三種策略：做多 / 放空 / 多空合併
```

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install finlab -q

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/機器學習/ds_ml_stock/'
print(f'BASE: {BASE}')

In [ ]:
import finlab
from google.colab import userdata
finlab_token = userdata.get("finlab")
finlab.login(finlab_token)

In [ ]:
finlab.login('YOUR_API_KEY_HERE')

## 重建模型架構

必須與 training.ipynb 中的定義完全一致，才能正確載入權重。

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_cont: int, cat_cardinalities: list, d_token: int):
        super().__init__()
        self.cont_W = nn.Parameter(torch.empty(n_cont, d_token))
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d_token))
        nn.init.kaiming_uniform_(self.cont_W, a=math.sqrt(5))
        self.cat_emb = nn.ModuleList([
            nn.Embedding(card, d_token) for card in cat_cardinalities
        ])

    def forward(self, x_cont, x_cat):
        t_cont = x_cont.unsqueeze(-1) * self.cont_W + self.cont_b
        t_cat  = torch.stack(
            [self.cat_emb[i](x_cat[:, i]) for i in range(x_cat.shape[1])], dim=1
        )
        return torch.cat([t_cont, t_cat], dim=1)


class FTTransformer(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, d_token=192,
                 n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token), nn.Linear(d_token, d_token // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_token // 2, 1),
        )

    def forward(self, x_cont, x_cat):
        tokens = self.tokenizer(x_cont, x_cat)
        cls    = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out    = self.transformer(tokens)
        return self.head(out[:, 0]).squeeze(-1)

def load_model(path: str, device) -> tuple:
    """載入儲存的模型，回傳 (model, cont_cols, bins_cols, config)。"""
    ckpt = torch.load(path, map_location=device)

    cont_cols        = ckpt['cont_cols']
    bins_cols        = ckpt['bins_cols']
    cat_cardinalities = ckpt['cat_cardinalities']
    config           = ckpt['config']

    model = FTTransformer(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cardinalities,
        d_token=config['d_token'],
        n_heads=config['n_heads'],
        n_layers=config['n_layers'],
        dropout=config['dropout'],
    ).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    return model, cont_cols, bins_cols


model_top, cont_cols, bins_cols = load_model(BASE + 'model_top.pt', DEVICE)
model_bot, _,         _         = load_model(BASE + 'model_bottom.pt', DEVICE)
print('✅ 模型載入完成')

In [ ]:
# 特徵資料（全期，含訓練 / 驗證 / 測試年份）
X = pd.read_parquet(BASE + 'X_features.parquet')

cont_indices = [X.columns.get_loc(c) for c in cont_cols]
cat_indices  = [X.columns.get_loc(c) for c in bins_cols]

print(f'特徵資料: {X.shape}')
print(f'日期範圍: {X.index.get_level_values("datetime").min()} ~ '
      f'{X.index.get_level_values("datetime").max()}')

In [ ]:
# 特徵資料（全期，含訓練 / 驗證 / 測試年份）
X = pd.read_parquet('X_features.parquet')

cont_indices = [X.columns.get_loc(c) for c in cont_cols]
cat_indices  = [X.columns.get_loc(c) for c in bins_cols]

print(f'特徵資料: {X.shape}')
print(f'日期範圍: {X.index.get_level_values("datetime").min()} ~ '
      f'{X.index.get_level_values("datetime").max()}')

## 全期推論

對所有 rev_date 的股票計算預測機率。

In [ ]:
@torch.no_grad()
def predict_proba(model: nn.Module, X_df: pd.DataFrame,
                  cont_indices: list, cat_indices: list,
                  device, batch_size: int = 1024) -> np.ndarray:
    model.eval()
    X_np = X_df.values.astype(float)
    probs = []
    for i in range(0, len(X_np), batch_size):
        xc = torch.FloatTensor(X_np[i:i+batch_size, cont_indices]).to(device)
        xk = torch.LongTensor(X_np[i:i+batch_size, cat_indices].astype(int)).to(device)
        p  = torch.sigmoid(model(xc, xk)).cpu().numpy()
        probs.append(p)
    return np.concatenate(probs)


probs_top = predict_proba(model_top, X, cont_indices, cat_indices, DEVICE)
probs_bot = predict_proba(model_bot, X, cont_indices, cat_indices, DEVICE)

print(f'推論完成：{len(probs_top):,} 筆')
print(f'Top model  機率分布: mean={probs_top.mean():.4f}  max={probs_top.max():.4f}')
print(f'Bottom model 機率分布: mean={probs_bot.mean():.4f}  max={probs_bot.max():.4f}')

`pd.Series(probs, index=X.index).unstack('instrument')` 將 MultiIndex Series
轉成 `(datetime × instrument)` 格式，供 `backtest.sim()` 使用。

In [ ]:
# 轉成 (datetime × instrument) 寬表格
prob_top_wide = FinlabDataFrame(
    pd.Series(probs_top, index=X.index).unstack('instrument')
)
prob_bot_wide = FinlabDataFrame(
    pd.Series(probs_bot, index=X.index).unstack('instrument')
)

print(f'prob_top_wide shape: {prob_top_wide.shape}  (rev_dates × stocks)')

In [ ]:
N_STOCKS = 10  # 每期做多 / 放空的股票數量，可調整

# 做多：top model 機率最高的 N 支（正值 = 做多）
position_long  = prob_top_wide.is_largest(N_STOCKS).astype(float)

# 放空：bottom model 機率最高的 N 支（負值 = 放空）
position_short = -prob_bot_wide.is_largest(N_STOCKS).astype(float)

# 多空合併：同一個 position DataFrame，正值做多、負值放空
position_ls = position_long + position_short

print(f'平均每期做多股數: {position_long.sum(axis=1).mean():.1f}')
print(f'平均每期放空股數: {(-position_short).sum(axis=1).mean():.1f}')

## 回測

- `resample=None`：只在 position 改變時換倉（自然對齊 rev_dates）
- `trade_at_price='open'`：以隔日開盤價成交，避免收盤價 lookahead
- `upload=False`：不上傳到 FinLab 雲端

In [ ]:
SIM_KWARGS = dict(
    resample        = None,
    trade_at_price  = 'open',
    fee_ratio       = 1.425 / 1000,
    tax_ratio       = 3 / 1000,
    upload          = False,
)

print('=== 做多策略 ===')
report_long  = backtest.sim(position_long,  name='Long Only',    **SIM_KWARGS)

print('\n=== 放空策略 ===')
report_short = backtest.sim(position_short, name='Short Only',   **SIM_KWARGS)

print('\n=== 多空合併 ===')
report_ls    = backtest.sim(position_ls,    name='Long-Short',   **SIM_KWARGS)

## 績效比較

In [ ]:
def extract_stats(report, name: str) -> dict:
    s = report.get_stats()
    m = report.get_metrics()
    trades = report.get_trades()
    r = trades['return']
    wins   = r[r > 0]
    losses = r[r < 0]
    avg_win  = wins.mean()   if len(wins)   > 0 else 0.0
    avg_loss = losses.mean() if len(losses) > 0 else 0.0
    rr      = avg_win / abs(avg_loss) if avg_loss != 0 else float('nan')
    payoff  = avg_win / abs(avg_loss) if avg_loss != 0 else float('nan')
    return {
        'Strategy'    : name,
        'CAGR'        : f"{s['cagr']:.2%}",
        'Sharpe'      : f"{s['monthly_sharpe']:.2f}",
        'Sortino'     : f"{m['ratio']['sortinoRatio']:.2f}",
        'Max Drawdown': f"{s['max_drawdown']:.2%}",
        'Win Rate'    : f"{s['win_ratio']:.2%}",
        'Total Return': f"{s['total_return']:.2%}",
        'RR'          : f"{rr:.2f}",
        '賺賠比'      : f"{payoff:.2f}",
    }

summary = pd.DataFrame([
    extract_stats(report_long,  'Long Only'),
    extract_stats(report_short, 'Short Only'),
    extract_stats(report_ls,    'Long-Short'),
]).set_index('Strategy')

display(summary)

In [ ]:
# 互動式報告（Jupyter 內直接顯示圖表）
print('=== 做多策略 ===')
report_long.display()

In [ ]:
print('=== 多空合併 ===')
report_ls.display()

## 訓練期 vs 測試期拆分績效

用 `live_performance_start` 參數讓報告自動拆分，
或手動篩選各期間的交易紀錄。

In [ ]:
# 測試年份（與 training.ipynb 一致）
TEST_YEARS = [2014, 2017, 2020, 2024]

def period_stats(report, years: list, label: str):
    trades = report.get_trades()
    if trades is None or trades.empty:
        print(f'{label}: 無交易紀錄')
        return
    mask = pd.to_datetime(trades['entry_date']).dt.year.isin(years)
    sub  = trades[mask]
    if sub.empty:
        print(f'{label}: 該期間無交易')
        return
    ret_col = 'return' if 'return' in sub.columns else sub.columns[-1]
    print(f'{label}')
    print(f'  交易次數 : {len(sub)}')
    print(f'  平均報酬 : {sub[ret_col].mean():.4%}')
    print(f'  勝率     : {(sub[ret_col] > 0).mean():.2%}')

print('=== 做多策略 ===')
period_stats(report_long, TEST_YEARS, '  Test 期間')

print('\n=== 多空合併 ===')
period_stats(report_ls,   TEST_YEARS, '  Test 期間')